In [ ]:
weights_path_final_3_gap_dmt = "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/baseline_rejection_ensemble/prot-DM_time_binary_classificator_241002_3_GAP-014-0.764-0.740.pth"
weights_path_final_resnet18_dmft = "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/baseline_rejection_ensemble/prot-DM_time_binary_classificator_resnet18-003-0.993-0.993.pth"
weights_path_final_5_gap_ft =  "final_checkpoints/baseline_rejection_ensemble/prot-DM_time_binary_classificator_241002_5_GAP-060-0.973-0.948.pth"

r1_params = {
    'conv_mlp': {
        'path': '/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/baseline_rejection_ensemble/prot-run_embedding_3_GAP_conv_mlp_lr1.05e-05_wd0.00e+00_drop0.0_channels64_extraFalse_pool7_hidden64_worker2_trial3-042-0.712-0.635.pth',
        'kwargs': {'model_name': 'conv_mlp', 'cnn_channels': 64, 'extra_conv': False, 'pool_size': 7, 'hidden_dim': 64}
    }
}

r2_params = {
    'conv_mlp': {
        'path': "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/baseline_rejection_ensemble/prot-run_embedding_r2_conv_mlp_lr4.51e-05_wd0.00e+00_drop0.2_channels64_extraTrue_pool7_hidden128_worker13_trial6-035-0.825-0.842.pth",
        'kwargs': {'model_name': 'conv_mlp', 'cnn_channels': 64, 'extra_conv': True, 'pool_size': 7, 'hidden_dim': 128}
    }
}

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import training_models
import training
from embedding_processing_models import build_embedding_processing
from DMTimeShardDataset import DMTimeShardDataset

device = "cuda" if torch.cuda.is_available() else "cpu"

# ========== Load Models ==========
dataset_cfg = {
    'output_dir': '/cephfs/users/oleksjuk/MA/WP2-1/DM_time_dataset_creator/outputs', 
    'prefix': 'B0531+21_59000_48386'
}

print('Loading models...')
small_model = training_models.models_htable['DM_time_binary_classificator_241002_3_GAP'](256, mode='dmt', dropout=False, device=device).to(device)
small_model.load_state_dict(torch.load(weights_path_final_3_gap_dmt, map_location=device)['model_state_dict'])
small_model.eval()

mid_model = training_models.models_htable['DM_time_binary_classificator_241002_5_GAP'](256, mode='ft', dropout=False, device=device).to(device)
mid_model.load_state_dict(torch.load(weights_path_final_5_gap_ft, map_location=device)['model_state_dict'])
mid_model.eval()

large_model = training_models.models_htable['DM_time_binary_classificator_resnet18'](256, mode="dmft", dropout=False, device=device).to(device)
large_model.load_state_dict(torch.load(weights_path_final_resnet18_dmft, map_location=device)["model_state_dict"])
large_model.eval()

# ========== Load Rejectors ==========
def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            value = checkpoint.get(key)
            if isinstance(value, dict):
                return value
    return checkpoint

def get_rejector_state_dict(ckpt):
    state_dict = extract_state_dict(ckpt)
    if not isinstance(state_dict, dict):
        return state_dict
    clean_dict = {}
    for k, v in state_dict.items():
        if k.startswith("1.net."):
            clean_dict[k.replace("1.net.", "net.")] = v
        elif k.startswith("1."):
            clean_dict[k.replace("1.", "")] = v
        elif k.startswith("embedding_processing."):
            clean_dict[k.replace("embedding_processing.", "")] = v
    return clean_dict or state_dict

# R1 rejector
r1_ckpt_path = r1_params['conv_mlp']['path']
r1_kwargs = r1_params['conv_mlp']['kwargs']
r1_model, r1_hook = build_embedding_processing(in_channels=12, **r1_kwargs)
r1_model.to(device)
r1_model.eval()
r1_ckpt = torch.load(r1_ckpt_path, map_location=device)
r1_model.load_state_dict(get_rejector_state_dict(r1_ckpt))

# R2 rejector
r2_ckpt_path = r2_params['conv_mlp']['path']
r2_kwargs = r2_params['conv_mlp']['kwargs']
r2_model, r2_hook = build_embedding_processing(in_channels=12, **r2_kwargs)
r2_model.to(device)
r2_model.eval()
r2_ckpt = torch.load(r2_ckpt_path, map_location=device)
r2_model.load_state_dict(get_rejector_state_dict(r2_ckpt))

print('Loading validation dataset...')
pulse_val_dataset = DMTimeShardDataset(dataset_cfg, use_freq_time=True, split="val")
pulse_val_dataset.labels = training.label_encoding(pulse_val_dataset.labels.astype(object))

pulse_val_loader = DataLoader(
    pulse_val_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=2,
)

In [ ]:
# ========== Collect Predictions and Scores ==========
print('Computing predictions and rejector scores...')
all_r1_scores = []
all_r2_scores = []
all_small_preds = []
all_mid_preds = []
all_large_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(pulse_val_loader):
        labels = batch['label'].to(device)

        # f_small
        small_input = small_model.features(batch)
        if small_input.dim() == 3:
            small_input = small_input.unsqueeze(0)
        small_logits = small_model.classifier(small_input)
        small_pred = small_logits.argmax(dim=1)

        # R1 scores
        r1_features = getattr(small_model, r1_hook)(small_input)
        if r1_features.dim() == 3:
            r1_features = r1_features.unsqueeze(0)
        elif r1_hook == 'pooled_features' and r1_features.dim() == 1:
            r1_features = r1_features.unsqueeze(0)
        r1_logits = r1_model(r1_features)
        r1_scores = torch.softmax(r1_logits, dim=1)[:, 1].detach()

        # f_mid
        mid_input = mid_model.features(batch)
        if mid_input.dim() == 3:
            mid_input = mid_input.unsqueeze(0)
        mid_logits = mid_model.classifier(mid_input)
        mid_pred = mid_logits.argmax(dim=1)

        # R2 scores
        r2_features = getattr(mid_model, r2_hook)(mid_input)
        if r2_features.dim() == 3:
            r2_features = r2_features.unsqueeze(0)
        elif r2_hook == 'pooled_features' and r2_features.dim() == 1:
            r2_features = r2_features.unsqueeze(0)
        r2_logits = r2_model(r2_features)
        r2_scores = torch.softmax(r2_logits, dim=1)[:, 1].detach()

        # f_large
        large_input = large_model.features(batch)
        if large_input.dim() == 3:
            large_input = large_input.unsqueeze(0)
        large_logits = large_model.classifier(large_input)
        large_pred = large_logits.argmax(dim=1)

        all_r1_scores.append(r1_scores.cpu())
        all_r2_scores.append(r2_scores.cpu())
        all_small_preds.append(small_pred.cpu())
        all_mid_preds.append(mid_pred.cpu())
        all_large_preds.append(large_pred.cpu())
        all_labels.append(labels.cpu())

r1_scores_np = torch.cat(all_r1_scores).numpy()
r2_scores_np = torch.cat(all_r2_scores).numpy()
small_preds_np = torch.cat(all_small_preds).numpy()
mid_preds_np = torch.cat(all_mid_preds).numpy()
large_preds_np = torch.cat(all_large_preds).numpy()
labels_np = torch.cat(all_labels).numpy()

print(f"Collected {len(labels_np)} samples")

# Latenz und Validierungsgenauigkeit über den Reject-Quoten

Die beiden finalen PDF-Abbildungen sind nativ für eine Einbindung mit `width=0.48\linewidth` ausgelegt.


In [ ]:
# ==========================================
# Thesis plotting setup
# ==========================================
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import PercentFormatter, FuncFormatter

OUTPUT_DIR = Path("thesis_plots_latency_accuracy")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Native width ~= 0.48 * thesis text width (~6.06 in).
# Thus the PDFs can be included with width=0.48\linewidth
# without materially rescaling the fonts.
FIG_HALF = (2.90, 2.35)

mpl.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "lines.linewidth": 0.8,
    "lines.markersize": 3.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PERCENT_0_1 = PercentFormatter(xmax=1.0, decimals=0)
COMMA_1 = FuncFormatter(lambda x, pos: f"{x:.1f}".replace(".", ","))


def save_pdf(fig, filename):
    path = OUTPUT_DIR / filename

    # No bbox_inches="tight": preserve the native physical width,
    # so 0.48\linewidth in LaTeX keeps the intended font size.
    fig.savefig(
        path,
        format="pdf",
    )

    print(f"Gespeichert: {path.resolve()}")
    return path


def style_rate_axes(ax):
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)

    ax.set_xlabel(
        r"$r_1$-Reject-Quote",
        labelpad=2,
    )
    ax.set_ylabel(
        r"$r_2$-Reject-Quote",
        labelpad=2,
    )

    ax.xaxis.set_major_formatter(PERCENT_0_1)
    ax.yaxis.set_major_formatter(PERCENT_0_1)

    ax.set_xticks(np.linspace(0, 1, 6))
    ax.set_yticks(np.linspace(0, 1, 6))

    ax.grid(False)


def style_rate_axes_3d(ax, zlabel, z_formatter=None):
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)

    ax.set_xlabel(
        r"$r_1$-Reject-Quote",
        labelpad=2,
    )
    ax.set_ylabel(
        r"$r_2$-Reject-Quote",
        labelpad=2,
    )
    ax.set_zlabel(
        zlabel,
        labelpad=4,
    )

    ax.xaxis.set_major_formatter(PERCENT_0_1)
    ax.yaxis.set_major_formatter(PERCENT_0_1)
    if z_formatter is not None:
        ax.zaxis.set_major_formatter(z_formatter)

    ax.set_xticks(np.linspace(0, 1, 6))
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.tick_params(axis="both", labelsize=7, pad=0)
    ax.tick_params(axis="z", labelsize=7, pad=1)
    ax.view_init(elev=25, azim=-135)
    ax.grid(True)


In [ ]:
# ==========================================
# Accuracy surface in reject-rate space
# + calibrated operating point
# ==========================================

# Number of threshold samples used to construct the accuracy surface.
# Increase if a denser surface is desired.
N_THRESHOLDS = 31

r1_thresholds = np.linspace(0.0, 1.0, N_THRESHOLDS)
r2_thresholds = np.linspace(0.0, 1.0, N_THRESHOLDS)

accuracy_grid = np.zeros(
    (len(r1_thresholds), len(r2_thresholds)),
    dtype=float,
)

r1_reject_rate_grid = np.zeros_like(accuracy_grid)
r2_reject_rate_grid = np.zeros_like(accuracy_grid)

print("Computing accuracy and reject-rate grids...")

for i, r1_t in enumerate(r1_thresholds):
    # Important: use >= consistently for all routing decisions.
    r1_reject_mask = r1_scores_np >= r1_t
    x_1 = float(r1_reject_mask.mean())

    for j, r2_t in enumerate(r2_thresholds):
        if np.any(r1_reject_mask):
            r2_conditional_reject_mask = (
                r2_scores_np[r1_reject_mask] >= r2_t
            )
            x_2 = float(r2_conditional_reject_mask.mean())
        else:
            x_2 = 0.0

        r2_reject_mask = (
            r1_reject_mask
            & (r2_scores_np >= r2_t)
        )
        r2_accept_mask = (
            r1_reject_mask
            & ~r2_reject_mask
        )

        y_pred = small_preds_np.copy()
        y_pred[r2_accept_mask] = mid_preds_np[r2_accept_mask]
        y_pred[r2_reject_mask] = large_preds_np[r2_reject_mask]

        accuracy_grid[i, j] = (
            y_pred == labels_np
        ).mean()

        r1_reject_rate_grid[i, j] = x_1
        r2_reject_rate_grid[i, j] = x_2

print("Grid computed.")


def threshold_for_reject_rate(scores, target_reject_rate):
    scores = np.asarray(scores, dtype=float).reshape(-1)

    if scores.size == 0:
        raise ValueError("scores must be non-empty")

    if not 0.0 <= target_reject_rate <= 1.0:
        raise ValueError("target_reject_rate must be in [0, 1]")

    threshold = float(
        np.quantile(
            scores,
            1.0 - target_reject_rate,
        )
    )

    achieved = float(
        (scores >= threshold).mean()
    )

    return threshold, achieved


# Final baseline routing quotas: 30% rejected by r1 and,
# conditionally, 30% rejected by r2.
TARGET_R1_REJECT_RATE = 0.30
TARGET_R2_REJECT_RATE = 0.30

selected_r1_threshold, selected_x1 = threshold_for_reject_rate(
    r1_scores_np,
    TARGET_R1_REJECT_RATE,
)

selected_r1_reject_mask = (
    r1_scores_np >= selected_r1_threshold
)

selected_r2_threshold, selected_x2 = threshold_for_reject_rate(
    r2_scores_np[selected_r1_reject_mask],
    TARGET_R2_REJECT_RATE,
)

selected_r2_reject_mask = (
    selected_r1_reject_mask
    & (r2_scores_np >= selected_r2_threshold)
)

selected_r2_accept_mask = (
    selected_r1_reject_mask
    & ~selected_r2_reject_mask
)

selected_prediction = small_preds_np.copy()
selected_prediction[selected_r2_accept_mask] = (
    mid_preds_np[selected_r2_accept_mask]
)
selected_prediction[selected_r2_reject_mask] = (
    large_preds_np[selected_r2_reject_mask]
)

selected_accuracy = float(
    (selected_prediction == labels_np).mean()
)

# Correct latency model:
# l(x1, x2) = 0.54 + 1.24*x1 + 2.38*x1*x2
selected_latency = (
    0.54
    + 1.24 * selected_x1
    + 2.38 * selected_x1 * selected_x2
)

print("\nSelected operating point")
print(f"r1 threshold:       {selected_r1_threshold:.6f}")
print(f"r1 reject rate:     {selected_x1:.4%}")
print(f"r2 threshold:       {selected_r2_threshold:.6f}")
print(f"r2 reject rate:     {selected_x2:.4%} (conditional on r1 reject)")
print(f"Validation accuracy:{selected_accuracy:.4%}")
print(f"Latency:            {selected_latency:.4f} ms")


## Validierungsgenauigkeit


In [ ]:
# ==========================================
# Plot 1: Validation accuracy vs reject rates
# ==========================================

# Reject-rate pairs are not located on a rectangular grid because
# r2 is evaluated conditionally on samples rejected by r1.
# Therefore use triangular interpolation in (x1, x2) space.

rate_points = np.column_stack((
    r1_reject_rate_grid.ravel(),
    r2_reject_rate_grid.ravel(),
))

accuracies = accuracy_grid.ravel()

# Remove exact duplicate rate pairs.
# Equal threshold intervals generally produce the same routing subset.
_, unique_idx = np.unique(
    rate_points,
    axis=0,
    return_index=True,
)

r1_rates_plot = rate_points[unique_idx, 0]
r2_rates_plot = rate_points[unique_idx, 1]
accuracy_plot = accuracies[unique_idx]

triangulation = mtri.Triangulation(
    r1_rates_plot,
    r2_rates_plot,
)

accuracy_levels = np.linspace(
    float(np.nanmin(accuracy_plot)),
    float(np.nanmax(accuracy_plot)),
    16,
)

# Globales Maximum der tatsächlich ausgewerteten Betriebspunkte
max_idx = int(np.nanargmax(accuracy_plot))
max_x1 = float(r1_rates_plot[max_idx])
max_x2 = float(r2_rates_plot[max_idx])
max_accuracy = float(accuracy_plot[max_idx])

fig = plt.figure(
    figsize=FIG_HALF,
    constrained_layout=True,
)
ax = fig.add_subplot(
    111,
    projection="3d",
)

surface = ax.plot_trisurf(
    triangulation,
    accuracy_plot,
    cmap="viridis",
    linewidth=0.15,
    edgecolor="0.35",
    alpha=0.95,
    antialiased=True,
)

# Hoechste tatsaechlich erreichte Validierungsgenauigkeit
ax.scatter(
    [max_x1],
    [max_x2],
    [max_accuracy],
    marker="*",
    s=55,
    color="black",
    edgecolors="white",
    linewidths=0.45,
    depthshade=False,
    zorder=6,
)

style_rate_axes_3d(
    ax,
    "Validierungsgenauigkeit",
    z_formatter=PERCENT_0_1,
)
ax.set_zlim(
    float(np.nanmin(accuracy_plot)),
    float(np.nanmax(accuracy_plot)),
)

cbar = fig.colorbar(
    surface,
    ax=ax,
    pad=0.06,
    fraction=0.06,
    shrink=0.72,
)

cbar.set_label(
    "Validierungsgenauigkeit",
    fontsize=8,
    labelpad=3,
)

cbar.ax.tick_params(
    labelsize=7,
)

cbar.ax.yaxis.set_major_formatter(
    PercentFormatter(
        xmax=1.0,
        decimals=0,
    )
)

save_pdf(
    fig,
    "validation_accuracy_reject_rates.pdf",
)

plt.show()

print(
    "Maximum:",
    f"x1={max_x1:.4f}, "
    f"x2={max_x2:.4f}, "
    f"accuracy={max_accuracy:.4%}",
)


## Latenz


In [ ]:
# ==========================================
# Plot 2: Latency vs reject rates
# ==========================================

# The latency model is defined directly in reject-rate space, so use
# a regular grid independent of the sampled rejector thresholds.
rate_values = np.linspace(
    0.0,
    1.0,
    201,
)

X1, X2 = np.meshgrid(
    rate_values,
    rate_values,
)

latency_surface = (
    0.54
    + 1.24 * X1
    + 2.38 * X1 * X2
)

latency_levels = np.linspace(
    float(latency_surface.min()),
    float(latency_surface.max()),
    16,
)

fig = plt.figure(
    figsize=FIG_HALF,
    constrained_layout=True,
)
ax = fig.add_subplot(
    111,
    projection="3d",
)

surface = ax.plot_surface(
    X1,
    X2,
    latency_surface,
    cmap="plasma",
    linewidth=0,
    antialiased=True,
    alpha=0.95,
)

style_rate_axes_3d(
    ax,
    "Latenz [ms]",
    z_formatter=COMMA_1,
)
ax.set_zlim(
    float(latency_surface.min()),
    float(latency_surface.max()),
)

cbar = fig.colorbar(
    surface,
    ax=ax,
    pad=0.06,
    fraction=0.06,
    shrink=0.72,
)

cbar.set_label(
    "Latenz [ms]",
    fontsize=8,
    labelpad=3,
)

cbar.ax.tick_params(
    labelsize=7,
)

cbar.ax.yaxis.set_major_formatter(
    FuncFormatter(
        lambda value, pos:
        f"{value:.1f}".replace(".", ",")
    )
)

save_pdf(
    fig,
    "latency_reject_rates.pdf",
)

plt.show()


In [ ]:
# ==========================================
# Optional: export operating point
# ==========================================

import pandas as pd

operating_point = pd.DataFrame([{
    "r1_threshold": selected_r1_threshold,
    "r1_reject_rate": selected_x1,
    "r2_threshold": selected_r2_threshold,
    "r2_conditional_reject_rate": selected_x2,
    "validation_accuracy": selected_accuracy,
    "latency_ms": selected_latency,
}])

display(operating_point)

operating_point.to_csv(
    OUTPUT_DIR / "operating_point.csv",
    index=False,
)
